<a href="https://colab.research.google.com/github/mowumialabi/west-africa-climate-trap-2026/blob/main/era5_thermodynamic_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import ee
import pandas as pd
import numpy as np

# =====================================================================
# GLOBAL CONFIGURATION: PROJECT SETTINGS & STATIONS
# =====================================================================
# Replace this placeholder string with your actual Google Cloud/GEE Project ID
GEE_PROJECT_ID = 'ee-mowumialabi'

print("Initializing Google Earth Engine Connection...")
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"Earth Engine successfully initialized with project: {GEE_PROJECT_ID}")
except Exception as e:
    print("Authentication required or initialization failed. Launching authorization flow...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"Earth Engine successfully authenticated and initialized with project: {GEE_PROJECT_ID}")

# Define geographic parameters globally across the West African transect
sites = {
    "Ile-Ife": {"lat": 7.484, "lon": 4.484},   # Core Southwest Moisture Trap
    "Abuja":   {"lat": 9.076, "lon": 7.398},   # Mid-Belt Transition Front
    "Kano":    {"lat": 12.002, "lon": 8.592}   # Northern Arid Boundary
}

start_date = "2026-01-01"
end_date = "2026-02-01"

# Define mathematical physics loop globally to support downstream calculation cells
def apply_magnus_tetens_physics(row):
    """
    Applies standard Magnus-Tetens coefficients over liquid water (T > 0 °C)
    dynamically handling both baseline and peak-hour data columns.
    """
    c1, c2, c3 = 0.61078, 17.27, 237.3  # Constants for kPa and °C

    # Check for column naming formats dynamically
    t_air = row["Air_Temp_Baseline_C"] if "Air_Temp_Baseline_C" in row else row["Air_Temp_C"]
    t_dew = row["Dew_Point_Baseline_C"] if "Dew_Point_Baseline_C" in row else row["Dew_Point_C"]

    # 1. Saturation Vapor Pressure (es) at Air Temp
    es = c1 * np.exp((c2 * t_air) / (c3 + t_air))

    # 2. Actual Vapor Pressure (ea) at Dew Point Temp
    ea = c1 * np.exp((c2 * t_dew) / (c3 + t_dew))

    # 3. Derive Relative Humidity and Vapor Pressure Deficit (VPD)
    rh = (ea / es) * 100.0
    vpd = es - ea

    return pd.Series([es, ea, vpd, rh])


# =====================================================================
# SECTION 1: 24-HOUR DIURNAL METEOROLOGICAL BASELINES (MANUSCRIPT 1)
# =====================================================================
print("\n" + "-"*70)
print("RUNNING SECTION 1: PROCESSING 24-HOUR CONTINUOUS BASELINES")
print("-"*70)
print("Ingesting continuous 24-hour monthly mean baseline arrays from ECMWF...")

# Pulling complete diurnal profiles across all 24 hours for January 2026
era5_baseline = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
                 .filterDate(start_date, end_date)
                 .select(["temperature_2m", "dewpoint_temperature_2m"]))

# Compute the absolute temporal mean across all hourly matrix columns
mean_baseline_layer = era5_baseline.mean()

extracted_baseline_records = []
print("Extracting baseline atmospheric spatial columns across the transect...")

for name, coords in sites.items():
    point = ee.Geometry.Point([coords["lon"], coords["lat"]])

    # Reduce region to extract spatial pixel values at native 9km grid scale
    stats = mean_baseline_layer.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=9000
    ).getInfo()

    # Thermodynamic Conversion: Kelvin to Celsius
    t_air_base_c = stats["temperature_2m"] - 273.15
    t_dew_base_c = stats["dewpoint_temperature_2m"] - 273.15

    extracted_baseline_records.append({
        "Site": name,
        "Latitude": f"{coords['lat']}°N",
        "Air_Temp_Baseline_C": t_air_base_c,
        "Dew_Point_Baseline_C": t_dew_base_c
    })

# Structure the output data matrix
df_baseline = pd.DataFrame(extracted_baseline_records)

print("Applying baseline Magnus-Tetens thermodynamic formulations...")
df_baseline[["es_base_kPa", "ea_base_kPa", "VPD_base_kPa", "RH_base_Pct"]] = df_baseline.apply(apply_magnus_tetens_physics, axis=1)

print("\n" + "="*70)
print("=== SECTION 1 RESULTS: PERSISTENT BACKGROUND METEOROLOGICAL MATRIX ===")
print("="*70)
print(df_baseline.round({
    "Air_Temp_Baseline_C": 1, "Dew_Point_Baseline_C": 1,
    "es_base_kPa": 4, "ea_base_kPa": 4, "VPD_base_kPa": 4, "RH_base_Pct": 1
}).to_string(index=False))


Initializing Google Earth Engine Connection...
Earth Engine successfully initialized with project: ee-mowumialabi

----------------------------------------------------------------------
RUNNING SECTION 1: PROCESSING 24-HOUR CONTINUOUS BASELINES
----------------------------------------------------------------------
Ingesting continuous 24-hour monthly mean baseline arrays from ECMWF...
Extracting baseline atmospheric spatial columns across the transect...
Applying baseline Magnus-Tetens thermodynamic formulations...

=== SECTION 1 RESULTS: PERSISTENT BACKGROUND METEOROLOGICAL MATRIX ===
   Site Latitude  Air_Temp_Baseline_C  Dew_Point_Baseline_C  es_base_kPa  ea_base_kPa  VPD_base_kPa  RH_base_Pct
Ile-Ife  7.484°N                 27.8                  20.5       3.7351       2.4078        1.3273         64.5
  Abuja  9.076°N                 29.1                   7.5       4.0192       1.0340        2.9852         25.7
   Kano 12.002°N                 24.4                  -0.2       3.